In [0]:
# pandas and pyspark are pre-installed core packages on Databricks — no install needed.

In [0]:
with open("/dbfs/databricks-datasets/tpch/README.md", "r") as f:
    content = f.read()
    displayHTML(f"<pre>{content}</pre>")

In [0]:
spark.sql("DROP DATABASE IF EXISTS bronze_layer CASCADE")
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_layer")

In [0]:
from pyspark.sql.functions import current_timestamp
from tqdm import tqdm

tpch_tables = [
    "customer", "lineitem", "nation", "orders", 
    "part", "partsupp", "region", "supplier"
]

def load_tables(tables: list[str], database: str):
    """
    Load tables from the tpch database to the bronze layer
    Args:
        tables (list[str]): list of tables to load
        database (str): database name
    """
    for table in tqdm(tables, desc="Loading tables"):
        print(f" | Current table: {table}")
        df = spark.read.table(f"samples.tpch.{table}")

        df_bronze = df.withColumn("ingestion_timestamp", current_timestamp())

        df_bronze.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(f"{database}.{table}")

load_tables(tpch_tables, "bronze_layer")
